In [4]:
import os
import glob
import re
import pandas as pd
import numpy as np
import xgboost as xgb
import joblib
from sklearn.ensemble import IsolationForest

np.random.seed(42)

# ==============================================================================
# 0. GLOBAL PATHS & SETUP
# ==============================================================================
BASE_DIR = r"C:\Users\phyot\Documents\SP\Hackathon\nebula x\02_Datasets"
OUTPUT_DIR = r"C:\Users\phyot\Documents\SP\Hackathon\nebula x\LTA_PS3_Submission\predictions"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==============================================================================
# 1. SUBSYSTEM: DOOR (Temporal Segment Detection)
# ==============================================================================
def parse_door_datetime(datetime_series: pd.Series) -> pd.Series:
    time_str = datetime_series.astype(str).str.split('-').apply(lambda x: '-'.join(x[:6]))
    return pd.to_datetime(time_str, format='%Y-%m-%d-%H-%M-%S', errors='coerce')

def run_door_pipeline():
    print("Running Door Pipeline...")
    test_file = os.path.join(BASE_DIR, "Door", "Test.csv")
    
    if not os.path.exists(test_file):
        print("  -> Skipping: Test.csv not found.")
        return None
        
    door_df = pd.read_csv(test_file)
    door_df['parsed_time'] = parse_door_datetime(door_df['Datetime'])
    door_df = door_df.sort_values(by='parsed_time').reset_index(drop=True)
    
    active_mask = (door_df['Open command'] == 1) | (door_df['Close command'] == 1) | (door_df['Door is opening'] == 1)
    door_df['segment_id'] = (~active_mask).cumsum()
    segments = door_df[active_mask].groupby('segment_id')
    
    results = []
    for _, seg in segments:
        start_time = seg['parsed_time'].min().strftime('%Y-%m-%d %H:%M:%S')
        end_time = seg['parsed_time'].max().strftime('%Y-%m-%d %H:%M:%S')
        
        max_current = seg['Motor current(mA)'].max()
        prediction = "Abnormal resistance" if max_current > 800 else "Normal"
        
        results.append({'start_time': start_time, 'end_time': end_time, 'prediction': prediction})
        
    preds_df = pd.DataFrame(results)
    preds_df.to_csv(os.path.join(OUTPUT_DIR, "door_predictions.csv"), index=False)
    print(f"  -> Exported door_predictions.csv ({len(preds_df)} segments found)")
    return None

# ==============================================================================
# 2. SUBSYSTEM: ACV (Fault Localisation)
# ==============================================================================
def run_acv_pipeline():
    print("Running ACV Pipeline...")
    test_dir = os.path.join(BASE_DIR, "ACV", "Test")
    test_files = glob.glob(os.path.join(test_dir, "*.csv")) + glob.glob(os.path.join(test_dir, "*.xlsx"))
    
    if not test_files:
        print("  -> Skipping: No test files found.")
        return None

    results = []
    for file_path in test_files:
        file_id = os.path.basename(file_path)
        df = pd.read_csv(file_path) if file_path.endswith('.csv') else pd.read_excel(file_path)
        
        car_numbers = list(set(re.findall(r'Car (\d{2})', ' '.join(df.columns))))
        car_scores = {}
        
        for car in car_numbers:
            temp_cols = [c for c in df.columns if f'Car {car}' in c and 'Temperatu' in c]
            if temp_cols:
                temp_data = df[temp_cols].apply(pd.to_numeric, errors='coerce')
                car_scores[car] = temp_data.var().sum()
            else:
                car_scores[car] = 0.0

        ranked_cars = sorted(car_scores, key=car_scores.get, reverse=True)
        results.append({'file_id': file_id, 'ranked_cars': "|".join(ranked_cars)})
        
    preds_df = pd.DataFrame(results)
    preds_df.to_csv(os.path.join(OUTPUT_DIR, "acv_predictions.csv"), index=False)
    print(f"  -> Exported acv_predictions.csv ({len(preds_df)} files processed)")
    return None

# ==============================================================================
# 3. SUBSYSTEM: RAIL CORRUGATION
# ==============================================================================
def run_rail_pipeline():
    print("Running Rail Corrugation Pipeline...")
    train_dir = os.path.join(BASE_DIR, "Rail_Corrugation", "Train")
    test_dir = os.path.join(BASE_DIR, "Rail_Corrugation", "Test")
    labels_file = os.path.join(BASE_DIR, "Rail_Corrugation", "Train_Labels.csv")
    
    if not os.path.exists(labels_file):
        print("  -> Skipping: Train_Labels.csv not found.")
        return None

    labels_df = pd.read_csv(labels_file)
    label_map = {0: 'Normal', 1: 'Side I', 2: 'Side II'}
    reverse_map = {'Normal': 0, 'Side I': 1, 'Side II': 2}
    
    def extract_features(df):
        num_df = df.select_dtypes(include=[np.number])
        fft_peaks = np.abs(np.fft.rfft(num_df.values, axis=0)).max(axis=0)
        return np.hstack([num_df.std().values, fft_peaks])

    train_files = glob.glob(os.path.join(train_dir, "*.csv"))
    X_train, y_train = [], []
    
    for f in train_files:
        file_id = os.path.basename(f)
        label_row = labels_df[labels_df.iloc[:, 0] == file_id]
        if not label_row.empty:
            label_str = label_row.iloc[0, 1]
            df = pd.read_csv(f)
            X_train.append(extract_features(df))
            y_train.append(reverse_map.get(label_str, 0))
            
    model = None
    if X_train:
        model = xgb.XGBClassifier(random_state=42)
        model.fit(np.array(X_train), np.array(y_train))
        
        test_files = glob.glob(os.path.join(test_dir, "*.csv"))
        results = []
        for f in test_files:
            file_id = os.path.basename(f)
            df = pd.read_csv(f)
            feat = extract_features(df).reshape(1, -1)
            pred_idx = model.predict(feat)[0]
            results.append({'file_id': file_id, 'prediction': label_map[pred_idx]})
            
        preds_df = pd.DataFrame(results)
        preds_df.to_csv(os.path.join(OUTPUT_DIR, "rail_predictions.csv"), index=False)
        print(f"  -> Exported rail_predictions.csv ({len(preds_df)} files processed)")
        
    return model

# ==============================================================================
# 4. SUBSYSTEM: SHM
# ==============================================================================
def run_shm_pipeline():
    print("Running SHM Pipeline...")
    train_dir = os.path.join(BASE_DIR, "SHM", "Train")
    test_dir = os.path.join(BASE_DIR, "SHM", "Test")
    labels_file = os.path.join(BASE_DIR, "SHM", "Train_Labels.csv")
    
    if not os.path.exists(labels_file):
        print("  -> Skipping: Train_Labels.csv not found.")
        return None

    labels_df = pd.read_csv(labels_file)
    
    def extract_features(df):
        stress_series = df.iloc[:, 0]
        return np.array([stress_series.mean(), stress_series.std(), stress_series.max() - stress_series.min(), np.sqrt((stress_series ** 2).mean())])

    train_files = glob.glob(os.path.join(train_dir, "*.csv"))
    X_train, y_train = [], []
    
    for f in train_files:
        file_id = os.path.basename(f)
        label_row = labels_df[labels_df.iloc[:, 0] == file_id]
        if not label_row.empty:
            target_val = float(label_row.iloc[0, 1])
            df = pd.read_csv(f, header=None)
            X_train.append(extract_features(df))
            y_train.append(target_val)
            
    model = None
    if X_train:
        model = xgb.XGBRegressor(n_estimators=100, max_depth=5, random_state=42)
        model.fit(np.array(X_train), np.array(y_train))
        
        test_files = glob.glob(os.path.join(test_dir, "*.csv"))
        results = []
        for f in test_files:
            file_id = os.path.basename(f)
            df = pd.read_csv(f, header=None)
            feat = extract_features(df).reshape(1, -1)
            pred_val = float(model.predict(feat)[0])
            results.append({'file_id': file_id, 'prediction': pred_val})
            
        preds_df = pd.DataFrame(results)
        preds_df.to_csv(os.path.join(OUTPUT_DIR, "shm_predictions.csv"), index=False)
        print(f"  -> Exported shm_predictions.csv ({len(preds_df)} files processed)")
        
    return model

# ==============================================================================
# 5. MODEL EXPORT UTILITY
# ==============================================================================
def export_trained_models(rail_model=None, shm_model=None, anomaly_model=None):
    model_export_dir = r"C:\Users\phyot\Documents\SP\Hackathon\nebula x\saved_models"
    os.makedirs(model_export_dir, exist_ok=True)
    
    print(f"\nExporting models to: {model_export_dir}")
    
    if rail_model is not None:
        rail_path = os.path.join(model_export_dir, "rail_corrugation_xgb.json")
        rail_model.save_model(rail_path)
        print(f"  ✓ Saved Rail Corrugation model")
        
    if shm_model is not None:
        shm_path = os.path.join(model_export_dir, "shm_xgb.json")
        shm_model.save_model(shm_path)
        print(f"  ✓ Saved SHM model")
        
    if anomaly_model is not None:
        anomaly_path = os.path.join(model_export_dir, "door_anomaly_forest.pkl")
        joblib.dump(anomaly_model, anomaly_path)
        print(f"  ✓ Saved Anomaly Detector")

# ==============================================================================
# 6. DIRECT EXECUTION BLOCK
# ==============================================================================
print("=== Nebula X: PS3 Model Training & Prediction ===")

# Run pipelines and capture models
run_door_pipeline()
run_acv_pipeline()
trained_rail_model = run_rail_pipeline()
trained_shm_model = run_shm_pipeline()

# Export captured models
export_trained_models(rail_model=trained_rail_model, shm_model=trained_shm_model)

print("\nAll operations complete. Zipping instructions:")
print(f"Please manually zip the contents of: {OUTPUT_DIR}")
print("Name the file 'predictions.zip' for submission.")

=== Nebula X: PS3 Model Training & Prediction ===
Running Door Pipeline...
  -> Exported door_predictions.csv (1 segments found)
Running ACV Pipeline...
  -> Exported acv_predictions.csv (1 files processed)
Running Rail Corrugation Pipeline...
  -> Exported rail_predictions.csv (68 files processed)
Running SHM Pipeline...
  -> Exported shm_predictions.csv (16 files processed)

Exporting models to: C:\Users\phyot\Documents\SP\Hackathon\nebula x\saved_models
  ✓ Saved Rail Corrugation model
  ✓ Saved SHM model

All operations complete. Zipping instructions:
Please manually zip the contents of: C:\Users\phyot\Documents\SP\Hackathon\nebula x\LTA_PS3_Submission\predictions
Name the file 'predictions.zip' for submission.
